# SQL reference 8: dates, times and time series

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-reference-08-dates-and-time-series.ipynb)

`DATE`, `TIMESTAMP` and `TIMESTAMP_NTZ` with the session time zone, parsing and formatting patterns, date arithmetic and differences, extracting and truncating, time-zone conversion and DST, and the time-series patterns: buckets, calendar spines, sessions, and cohorts and retention. Practise it in [SQL practice 8: dates, times and time series](sql-practice-08-dates-and-time-series.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and three helpers: `show(query)` runs a query and prints its rows untruncated, `plan(query)` returns what `EXPLAIN` prints, and `expected_error()` prints the error class of an example that fails on purpose.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Contents

| section | entries |
| --- | --- |
| [Date and time types](#date-and-time-types) | [`DATE`, `TIMESTAMP` and `TIMESTAMP_NTZ`](#date-timestamp-and-timestamp_ntz) |
| [Parsing and formatting](#parsing-and-formatting) | [`to_date`, `to_timestamp` and `date_format`](#to_date-to_timestamp-and-date_format) |
| [Date arithmetic](#date-arithmetic) | [Adding, subtracting and differences](#adding-subtracting-and-differences), [Extracting parts and truncating](#extracting-parts-and-truncating) |
| [Time zones](#time-zones) | [Converting between zones](#converting-between-zones) |
| [Time-series patterns](#time-series-patterns) | [Bucketing and fixed windows](#bucketing-and-fixed-windows), [Calendar spines and gap filling](#calendar-spines-and-gap-filling), [Sessions](#sessions), [Cohorts and retention](#cohorts-and-retention) |

## Date and time types

### `DATE`, `TIMESTAMP` and `TIMESTAMP_NTZ`

| type | represents | displayed in |
| --- | --- | --- |
| `DATE` | a calendar day | as is |
| `TIMESTAMP` (`TIMESTAMP_LTZ`) | an instant (stored as microseconds since the epoch, UTC) | the session time zone, `spark.sql.session.timeZone` |
| `TIMESTAMP_NTZ` | a wall-clock date and time with no zone | as is, never converted |

| expression | returns |
| --- | --- |
| `DATE'2026-01-05'`, `TIMESTAMP'2026-01-05 09:15:00'`, `TIMESTAMP_NTZ'...'` | literals |
| `current_date()`, `current_timestamp()`, `now()` | the query's start time (constant within a query) |
| `CAST(ts AS DATE)` / `to_date(ts)` | the date part in the session time zone |

**Notes:** the same `TIMESTAMP` value shows a different wall-clock time (and possibly date) under a different
session time zone. Pin `spark.sql.session.timeZone` (this notebook uses UTC) so results do not depend on the machine.
Snowflake's default `TIMESTAMP_NTZ` and BigQuery's `DATETIME` correspond to Spark's `TIMESTAMP_NTZ`.

In [ ]:
show("SELECT TIMESTAMP'2026-01-14 00:30:00' AS ts, to_date(TIMESTAMP'2026-01-14 00:30:00') AS utc_date, typeof(TIMESTAMP_NTZ'2026-01-14 00:30:00') AS ntz")
spark.conf.set("spark.sql.session.timeZone", "America/Los_Angeles")
show("SELECT order_ts, to_date(order_ts) AS local_date FROM orders WHERE order_id = 12")   # same instant, previous day in LA
spark.conf.set("spark.sql.session.timeZone", "UTC")

## Parsing and formatting

### `to_date`, `to_timestamp` and `date_format`

| function | does |
| --- | --- |
| `to_date(s [, fmt])`, `to_timestamp(s [, fmt])` | parse a string; error on a bad value in ANSI mode |
| `try_to_timestamp(s [, fmt])` | NULL on a bad value |
| `date_format(ts, fmt)` | format as a string |
| `make_date(y, m, d)`, `make_timestamp(y, mo, d, h, mi, s)` | build from parts |
| `unix_seconds(ts)`, `unix_millis(ts)`, `timestamp_seconds(n)`, `timestamp_millis(n)` | epoch conversions |
| `from_unixtime(n [, fmt])` | epoch seconds to a formatted string |

| pattern | meaning | pattern | meaning |
| --- | --- | --- | --- |
| `yyyy` | year | `HH` / `hh a` | hour 0-23 / 1-12 with AM-PM |
| `MM` / `MMM` / `MMMM` | month 01 / Jan / January | `mm`, `ss`, `SSS` | minute, second, millisecond |
| `dd` | day of month | `EEE` / `EEEE` | Fri / Friday |
| `D` | day of year | `XXX` / `z` | offset `+05:30` / zone name |

**Notes:** Spark uses Java `DateTimeFormatter` patterns: `mm` is minutes and `MM` months, a frequent bug. Postgres
and Snowflake use `YYYY-MM-DD HH24:MI:SS`-style templates, BigQuery uses `%Y-%m-%d` (strftime).

In [ ]:
show("""
SELECT to_date('05/01/2026', 'dd/MM/yyyy') AS parsed_date, to_timestamp('2026-01-05 9:15 PM', 'yyyy-MM-dd h:mm a') AS parsed_ts,
       try_to_timestamp('not a date') AS bad, date_format(TIMESTAMP'2026-01-05 21:15:00', 'EEE dd MMM yyyy HH:mm') AS formatted,
       make_date(2026, 2, 28) AS built, timestamp_millis(1767605700000) AS from_millis, unix_seconds(TIMESTAMP'2026-01-01 00:00:00') AS epoch
""")

## Date arithmetic

### Adding, subtracting and differences

| function | returns |
| --- | --- |
| `date_add(d, n)`, `date_sub(d, n)`, `d + INTERVAL 3 DAYS` | a date `n` days later / earlier |
| `ts + INTERVAL 90 MINUTES`, `timestampadd(HOUR, 2, ts)` | a shifted timestamp |
| `add_months(d, n)` | same day `n` months later, clamped to month end (`2026-01-31` + 1 = `2026-02-28`) |
| `datediff(end, start)` | whole days between two dates (end − start) |
| `timestampdiff(unit, start, end)` | whole units between timestamps (`MINUTE`, `HOUR`, `DAY`, `MONTH`, ...) |
| `months_between(end, start)` | fractional months |
| `last_day(d)`, `next_day(d, 'Mon')` | month end / next given weekday |

**Notes:** argument order differs between engines: Spark's `datediff(end, start)` matches MySQL, while Snowflake and
SQL Server write `datediff(unit, start, end)`. Subtracting two timestamps in Spark returns an interval; use
`timestampdiff` or `unix_seconds(a) - unix_seconds(b)` for numbers.

In [ ]:
show("""
SELECT date_add(DATE'2026-01-31', 1) AS plus_day, add_months(DATE'2026-01-31', 1) AS plus_month,
       datediff(DATE'2026-03-01', DATE'2026-02-01') AS days, timestampdiff(MINUTE, TIMESTAMP'2026-01-05 09:15:00', TIMESTAMP'2026-01-05 11:40:00') AS minutes,
       TIMESTAMP'2026-01-05 09:15:00' + INTERVAL 90 MINUTES AS later, last_day(DATE'2028-02-10') AS month_end, next_day(DATE'2026-01-05', 'Fri') AS next_friday
""")

### Extracting parts and truncating

| function | returns |
| --- | --- |
| `year`, `quarter`, `month`, `day`, `hour`, `minute`, `second` | parts as integers |
| `dayofweek(d)` | 1 = Sunday … 7 = Saturday (Spark and MySQL); `weekday(d)` is 0 = Monday |
| `weekofyear(d)` | ISO week number |
| `extract(field FROM d)` / `date_part('field', d)` | standard part extraction (`DOW`, `DOW_ISO` (Postgres: `ISODOW`), `WEEK`, `DOY`, ...) |
| `date_trunc('unit', ts)` | timestamp truncated to `year`, `quarter`, `month`, `week` (Monday), `day`, `hour`, `minute` |
| `trunc(d, 'unit')` | date truncated to year, quarter, month or week |

In [ ]:
show("""
SELECT order_id, order_ts, dayofweek(order_ts) AS dow, extract(DOW_ISO FROM order_ts) AS iso_dow, weekofyear(order_ts) AS week,
       date_trunc('week', order_ts) AS week_start, date_trunc('hour', order_ts) AS hour_start, trunc(order_ts, 'month') AS month_start
FROM orders WHERE order_id IN (1, 6, 12) ORDER BY order_id
""")

## Time zones

### Converting between zones

| function | does |
| --- | --- |
| `from_utc_timestamp(ts, zone)` | treat `ts` as UTC and return the wall-clock time in `zone` |
| `to_utc_timestamp(ts, zone)` | treat `ts` as wall-clock time in `zone` and return UTC |
| `convert_timezone([source,] target, ts_ntz)` | convert a `TIMESTAMP_NTZ` wall-clock time between zones |

**Notes:** use IANA zone names (`America/New_York`), never fixed abbreviations like `EST`, so daylight-saving time is
applied. Business reporting usually needs "the local date of the customer", which can differ from the UTC date near
midnight: order 12 at 00:30 UTC on the 14th is the evening of the 13th in New York.

In [ ]:
show("""
SELECT order_id, order_ts AS utc,
       from_utc_timestamp(order_ts, 'America/New_York') AS new_york,
       to_date(from_utc_timestamp(order_ts, 'America/New_York')) AS ny_date,
       from_utc_timestamp(order_ts, 'Asia/Kolkata') AS kolkata
FROM orders WHERE order_id IN (11, 12) ORDER BY order_id
""")
show("SELECT from_utc_timestamp(TIMESTAMP'2026-03-08 06:30:00', 'America/New_York') AS before_dst, from_utc_timestamp(TIMESTAMP'2026-03-08 07:30:00', 'America/New_York') AS after_dst")

## Time-series patterns

### Bucketing and fixed windows

| need | expression |
| --- | --- |
| calendar buckets | `date_trunc('hour' \| 'day' \| 'week' \| 'month', ts)` |
| fixed-width buckets (15 minutes) | `window(ts, '15 minutes').start`, or `timestamp_seconds(floor(unix_seconds(ts) / 900) * 900)` |
| sliding buckets | `window(ts, '1 hour', '15 minutes')`: each row falls in every overlapping window |

In [ ]:
show("""
SELECT window(ts, '1 hour').start AS hour_start, count(*) AS events
FROM events WHERE to_date(ts) = DATE'2026-01-06'
GROUP BY window(ts, '1 hour') ORDER BY hour_start
""")

### Calendar spines and gap filling

Aggregations only return periods that have data. To report every day (including zeros), generate a **spine** of all
periods and left join the facts to it. In Spark, `explode(sequence(start, end, INTERVAL 1 DAY))` builds the spine;
other engines use `generate_series` (Postgres, DuckDB), `GENERATOR` (Snowflake) or `GENERATE_DATE_ARRAY` (BigQuery).

In [ ]:
show("""
WITH spine AS (SELECT explode(sequence(DATE'2026-01-05', DATE'2026-01-14', INTERVAL 1 DAY)) AS day),
     daily AS (SELECT to_date(order_ts) AS day, sum(amount) AS revenue, count(*) AS orders FROM orders GROUP BY 1)
SELECT s.day, coalesce(d.orders, 0) AS orders, coalesce(d.revenue, 0) AS revenue,
       sum(coalesce(d.revenue, 0)) OVER (ORDER BY s.day ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS trailing_3_day
FROM spine s LEFT JOIN daily d USING (day)
ORDER BY s.day
""")

### Sessions

A **session** is a run of a user's events with no gap longer than a threshold (commonly 30 minutes). Compute it in
three steps: flag events that start a new session (first event, or gap > threshold, using `lag`), number sessions with
a running `sum` of the flags, then aggregate per session.

In [ ]:
show("""
WITH gaps AS (
    SELECT user_id, ts,
           CASE WHEN lag(ts) OVER (PARTITION BY user_id ORDER BY ts) IS NULL
                  OR ts > lag(ts) OVER (PARTITION BY user_id ORDER BY ts) + INTERVAL 30 MINUTES
                THEN 1 ELSE 0 END AS new_session
    FROM (SELECT DISTINCT user_id, ts FROM events)
),
numbered AS (SELECT *, sum(new_session) OVER (PARTITION BY user_id ORDER BY ts) AS session_no FROM gaps)
SELECT user_id, session_no, min(ts) AS started, max(ts) AS ended, count(*) AS events
FROM numbered GROUP BY user_id, session_no ORDER BY user_id, session_no
""")

### Cohorts and retention

A **cohort** groups users by when they started (first order month, signup week). Retention is the share of each
cohort that is active again *n* periods later: assign each user their cohort period, compute the period offset of
every later activity, then count distinct users per (cohort, offset) and divide by the cohort size.

In [ ]:
show("""
WITH activity AS (SELECT DISTINCT user_id, to_date(ts) AS day FROM events),
     cohorts AS (SELECT user_id, min(day) AS cohort_day FROM activity GROUP BY user_id),
     offsets AS (SELECT a.user_id, c.cohort_day, datediff(a.day, c.cohort_day) AS day_n
                 FROM activity a JOIN cohorts c USING (user_id))
SELECT cohort_day, count(DISTINCT user_id) AS users,
       count(DISTINCT CASE WHEN day_n BETWEEN 1 AND 3 THEN user_id END) AS returned_within_3_days
FROM offsets GROUP BY cohort_day ORDER BY cohort_day
""")

**Next:** practise these entries in [SQL practice 8: dates, times and time series](sql-practice-08-dates-and-time-series.ipynb).